# Test Qwen3.6-27B-FP8 — Domino / H100

Notebook de diagnostic à exécuter dans l'ordre :
1. configuration offline ;
2. environnement CUDA ;
3. modèle local ;
4. processor ;
5. chargement FP8 ;
6. génération texte ;
7. test vision sur PDF.


In [ ]:
# 1 — Configuration offline (avant import transformers)
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["USE_HUB_KERNELS"] = "0"
os.environ["TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR"] = "1"

for k in ["HF_HUB_OFFLINE","TRANSFORMERS_OFFLINE","USE_HUB_KERNELS","TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR"]:
    print(k, "=", os.environ[k])


In [ ]:
# 2 — Environnement
import sys, torch, transformers
print("Python       :", sys.version.split()[0])
print("Torch        :", torch.__version__)
print("CUDA Torch   :", torch.version.cuda)
print("Transformers :", transformers.__version__)
print("CUDA dispo   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM totale  :", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2), "GB")


In [ ]:
# 3 — Chemin local du modèle
from pathlib import Path
MODEL_PATH = Path("/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main")

print("MODEL_PATH :", MODEL_PATH)
print("Existe     :", MODEL_PATH.exists())

required_files = [
    "config.json","generation_config.json","preprocessor_config.json",
    "tokenizer_config.json","tokenizer.json","chat_template.jinja",
    "model.safetensors.index.json"
]
for f in required_files:
    print(f"{f:35} : {(MODEL_PATH/f).exists()}")
assert MODEL_PATH.exists()


In [ ]:
# 4 — Processor
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True
)
print("PROCESSOR OK")
print(type(processor))


In [ ]:
# 5 — Modèle
import torch
from transformers import AutoModelForMultimodalLM

print("Chargement Qwen3.6-27B-FP8...")
model = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
    device_map="auto",
    dtype="auto"
)
model.eval()

print("MODEL OK")
print("Type   :", type(model))
print("Device :", next(model.parameters()).device)
print("Dtype  :", next(model.parameters()).dtype)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM allouée :", round(torch.cuda.memory_allocated()/1024**3,2), "GB")


In [ ]:
# 6 — Test génération TEXTE
messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "Réponds uniquement par OK."}]
}]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k,v in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)

generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
result = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print("RESULTAT :", result)


## Test vision
Ne continuer que si la génération texte fonctionne. Modifier `PDF_TEST` ci-dessous.


In [ ]:
# 7 — Première page PDF -> image
import fitz, io
from PIL import Image
from IPython.display import display

PDF_TEST = "/mnt/data/ton_fichier_test.pdf"  # À MODIFIER
assert os.path.exists(PDF_TEST), f"PDF introuvable : {PDF_TEST}"

doc = fitz.open(PDF_TEST)
page = doc[0]
pix = page.get_pixmap(matrix=fitz.Matrix(2.0,2.0), alpha=False)
image = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
print("Pages :", len(doc))
print("Image :", image.size)
display(image)


In [ ]:
# 8 — Test VISION
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": '''Analyse cette page.
Retourne uniquement ce JSON :
{
  "type_document": null,
  "titre_detecte": null
}'''}
    ]
}]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k,v in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=150, do_sample=False)

generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
result = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print("===== RESULTAT VISION =====")
print(result)


In [ ]:
# 9 — État GPU final
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM allouée :", round(torch.cuda.memory_allocated()/1024**3,2), "GB")
    print("VRAM réservée:", round(torch.cuda.memory_reserved()/1024**3,2), "GB")
